In [1]:
import warnings
from pathlib import Path

import numpy as np
import prism

from imagematerials import buildings as bld
from imagematerials import vehicles as vhc
from imagematerials.concepts import create_building_graph, create_vehicle_graph
from imagematerials.factory import ModelFactory, Sector
from imagematerials.model import GenericMaterials, GenericStocks, MaterialIntensities
from imagematerials.preprocessing import get_preprocessing_data



In [2]:
bld_sector = get_preprocessing_data("buildings", Path("..", "data", "raw"), cache="prep_buildings.nc")
vhc_sector = get_preprocessing_data("vehicles", Path("..", "data", "raw"), cache="prep_vema.nc")


In [ ]:
def get_buildings_prep():
    circularity_scenario = "base"
    climate_scenario = 'IMAGE_CircoMod/SSP2'
    climate_policy_scenario_dir = Path("../data/raw") / climate_scenario
    scenario_path = Path("../data/raw") / 'circular_economy_scenarios' / circularity_scenario
    circular_economy_scenario_dirs = {circularity_scenario: scenario_path}
    climate_policy_config = read_climate_policy_config(climate_policy_scenario_dir)
    circular_economy_config = read_circular_economy_config(circular_economy_scenario_dirs)
    base_dir = Path("..", "data", "raw")
    prep_fp = Path("prep_buildings.nc")
    if not prep_fp.is_file():
        prep_data = bld.preprocess(base_dir, climate_policy_config, circular_economy_config)
        export_to_netcdf(prep_data, prep_fp)
    else:
        prep_data = import_from_netcdf(prep_fp)
    new_prep_data = {k: v for k, v in prep_data.items()}
    new_prep_data["knowledge_graph"] = create_building_graph()
    new_prep_data["shares"] = None
    sec_bld = Sector("bld", new_prep_data)
    return sec_bld

In [ ]:
sectors = [vhc_sector, bld_sector]
ns_coordinates = {
    "Region": sectors[0].coordinates["Region"],
    "material": list(set(sectors[0].coordinates["material"] + sectors[1].coordinates["material"]))
}
ns_combined = Sector("combi", {}, coordinates=ns_coordinates)
sectors.append(ns_combined)

In [4]:
# Define the complete timeline, including historic tail
# time_start = prep_data["stocks"].coords["Time"].min().values
time_start = 1960
complete_timeline = prism.Timeline(time_start, 2060, 1)
simulation_timeline = prism.Timeline(1970, 2060, 1)

In [5]:
REGION = prism.Dimension("Region")
MATERIAL_TYPE = prism.Dimension("material")

@prism.interface
class SumMaterials(prism.Model):
    Region: prism.Coords[REGION]
    material: prism.Coords[MATERIAL_TYPE]

    input_data: tuple[str] = ("inflow_materials", )
    output_data: tuple[str] = ("summed_inflow_materials", )



    summed_inflow_materials: prism.TimeVariable[REGION, MATERIAL_TYPE, "count"] = prism.export()

    def compute_initial_values(self, time: prism.Timeline):
        pass

    def compute_values(self, time: prism.Time, inflow_materials):
        t, dt = time.t, time.dt
        self.summed_inflow_materials[t].loc[:] = 0
        for inflow in inflow_materials:
            self.summed_inflow_materials[t].loc[{"material": inflow[t].coords["material"]}] += inflow[t].sum("Type")

In [6]:
factory = ModelFactory(
    sectors, complete_timeline
    ).add(GenericStocks, ["buildings", "vehicles"]
    ).add(GenericMaterials,  "vehicles"
    ).add(MaterialIntensities, "buildings",
    ).add(SumMaterials, "combi", input_sources={"inflow_materials": ["buildings", "vehicles"]}
    )
model = factory.finish()

In [7]:
factory.visualize(px_x="1000px", px_y="700px").show("model.html", notebook=True)

model.html


In [8]:
import warnings
with warnings.catch_warnings():
    warnings.filterwarnings("ignore")
    model.simulate(complete_timeline)